In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient
from transformers import AutoTokenizer

from data.scripts.shared.chunks_by_tokens import split_text_to_chunks
from data.scripts.shared.normailze_text import normalize_text
from data.scripts.shared.balance_df_authors import balance_authors

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:2000"
MLFLOW_EXPERIMENT_NAME = "combined_dataset"
MLFLOW_RUN_NAME = "second run"

TOKENIZER_NAME = 'DeepPavlov/rubert-base-cased'
RANDOM_STATE = 67

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.start_run(run_name=MLFLOW_RUN_NAME)

mlflow.log_param("socials_dataset_name", 'socials.csv')
mlflow.log_param("literature_dataset_name", 'balanced_lit.csv')
mlflow.log_param("npl1_dataset_name", 'npl1.csv')

mlflow.set_tag("upstream.ruslit_preprocess_run_id", '406f39f4f7844e10b58b1b508d6f2bc7')

client = MlflowClient()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
mlflow.log_param('tokenizer', TOKENIZER_NAME)

In [ ]:
socials_df = pd.read_csv('../../data/data/socials.csv')
socials_df.columns=["author","text"]
socials_df.head()

In [ ]:
local_path = client.download_artifacts(run_id="21969b89951c44b8885f35c231b29a94", path="data/socials_author_metrics.csv")

social_metrics = pd.read_csv(local_path)
social_metrics.head()

In [ ]:
ruslit_df = pd.read_csv('../../data/data/balanced_lit.csv', delimiter=',')
ruslit_df.head()

In [ ]:
nplus1_df = pd.read_csv('../../data/data/npl1.csv', delimiter=',')
nplus1_df.head()

In [ ]:
def visualise_mean_len(socials, ruslit, nplus1, suffix: str = None):
    socials['len'] = [len(x) for x in socials['text']]
    ruslit['len'] = [len(x) for x in ruslit['text']]
    nplus1['len'] = [len(x) for x in nplus1['text']]

    mean_text_lens = [
        {
            'df': 'socials_df',
            'len': socials['len'].mean(),
        },
        {
            'df': 'ruslit_df',
            'len': ruslit['len'].mean(),
        },
        {
            'df': 'scinews_df',
            'len': nplus1['len'].mean(),
        }
    ]
    mean_text_lens = pd.DataFrame(mean_text_lens)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax = sns.barplot(x='df', y='len',data=mean_text_lens)
    ax.bar_label(ax.containers[0])
    ax.set_title('Mean length of texts')
    fig.tight_layout()

    display(fig)

    mlflow.log_figure(fig, f"plots/mean_len_for_genre_{suffix if suffix else ''}.png")

    plt.close(fig)

In [ ]:
def visualise_df_size(socials, ruslit, nplus1, suffix: str = None):
    lens = [
        {
            'df': 'socials_df',
            'len':len(socials),
        },
        {
            'df': 'ruslit_df',
            'len':len(ruslit),
        },
        {
            'df': 'scinews_df',
            'len':len(nplus1),
        }
    ]
    lens = pd.DataFrame(lens)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax = sns.barplot(x='df', y='len',data=lens)
    ax.bar_label(ax.containers[0])
    ax.set_title('Sizes of datasets')
    fig.tight_layout()

    display(fig)

    mlflow.log_figure(fig, f"plots/mean_len_for_genre_{suffix if suffix else ''}.png")

    plt.close(fig)

In [ ]:
visualise_df_size(socials_df, ruslit_df, nplus1_df, 'no_preproc')

In [ ]:
visualise_mean_len(socials_df, ruslit_df, nplus1_df, 'no_preproc')

In [ ]:
percentile = 99.9
filter_metrics = [
    'avg_sent_len_words',
    'avg_word_len_chars',
    'mtld',
    'flesch_reading_ease',
    'gunning_fog',
]
social_metrics_filtered = social_metrics.copy()
for metric in filter_metrics:
    for suf in ['_median', '_mean']:

        threshold = social_metrics[metric + suf].quantile(percentile / 100)
        mask = social_metrics[metric + suf] <= threshold
        social_metrics_filtered = social_metrics_filtered[mask]

In [ ]:
socials_df = socials_df[socials_df.author.isin(social_metrics_filtered.author)]
socials_df.head()

In [ ]:
len(socials_df)

In [ ]:
socials_df["text"] = socials_df["text"].apply(
    lambda x:
    normalize_text(x, True)
)
nplus1_df["text"] = nplus1_df["text"].apply(
    lambda x:
    normalize_text(x, True)
)

In [ ]:
ruslit_df.dropna(how='any',axis=0, inplace=True)
nplus1_df.dropna(how='any',axis=0, inplace=True)
socials_df.dropna(how='any',axis=0, inplace=True)

In [ ]:
def aggregate_small_text(posts: list[str],
                                  target_tokens: int = 150,
                                  tokenizer=None) -> list[str]:
    if tokenizer is None:
        raise ValueError("Нужен токенизатор для контроля длины")

    aggregated_fragments = []
    shuffled_posts = posts.copy()
    random.shuffle(shuffled_posts)

    current_fragment = []
    current_length   = 0

    for post in shuffled_posts:
        if type(post) is not str:
            print(post)
        post_tokens = len(tokenizer.encode(post, add_special_tokens=False))

        if current_length + post_tokens >= target_tokens:
            if current_fragment:
                aggregated_fragments.append(' [SEP] '.join(current_fragment))
            current_fragment = [post]
            current_length   = post_tokens
        else:
            current_fragment.append(post)
            current_length += post_tokens

    if current_length >= target_tokens // 2:
        aggregated_fragments.append(' [SEP] '.join(current_fragment))

    return aggregated_fragments

In [ ]:
socials_aggregated = []
cnt = 0
for author, group in socials_df.groupby("author"):
    texts = aggregate_small_text(list(group['text']), 150, tokenizer=tokenizer)
    for text in texts:
        socials_aggregated.append(
            {'author': author, 'text': text}
        )


socials_agr_df = pd.DataFrame(socials_aggregated).set_index("author")
socials_agr_df.head()

In [ ]:
nplus1_aggregated = []
cnt = 0
for author, group in nplus1_df.groupby("author"):
    texts = aggregate_small_text(list(group['text']), 150, tokenizer=tokenizer)
    for text in texts:
        nplus1_aggregated.append(
            {'author': author, 'text': text}
        )


nplus1_aggregated_df = pd.DataFrame(nplus1_aggregated).set_index("author")
nplus1_aggregated_df.head()

In [ ]:
nplus1_split = split_text_to_chunks(nplus1_aggregated_df, tokenizer, chunk_size=100)
nplus1_split.head()

In [ ]:
socials_split = split_text_to_chunks(socials_agr_df, tokenizer)
socials_split.head()

In [ ]:
visualise_df_size(socials_split, ruslit_df, nplus1_split, 'agregated_and_splited')

In [ ]:
visualise_mean_len(socials_split, ruslit_df, nplus1_split, 'agregated_and_splited')

In [ ]:
balanced_socials = balance_authors(socials_split, min_fragments=150, max_fragments=500)
balanced_socials.head()

In [ ]:
balanced_ruslit = balance_authors(ruslit_df, min_fragments=100, max_fragments=500)
balanced_ruslit.head()

In [ ]:
balanced_nplus1 = balance_authors(nplus1_split, min_fragments=100, max_fragments=500)
balanced_nplus1.head()

In [ ]:
visualise_df_size(balanced_socials, balanced_ruslit, balanced_nplus1, 'balanced')

In [ ]:
from sklearn.model_selection import train_test_split

splited_ruslit, _ = train_test_split(balanced_ruslit, train_size=0.5, test_size=0.5)


In [ ]:
visualise_df_size(balanced_socials, splited_ruslit, balanced_nplus1, 'balanced')

In [ ]:
new_df_raw = []
for x in balanced_ruslit.iterrows():
    new_df_raw.append({
        'author': x[1]['author'],
        'text': x[1]['text'],
        'source_id': x[1]['source_id'],
        'source_type': 'ruslit',
    })
new_df_raw[0]

In [ ]:
for x in balanced_nplus1.iterrows():
    new_df_raw.append({
        'author': x[1]['author'],
        'text': x[1]['text'],
        'source_id': x[1]['source_id'],
        'source_type': 'nplus1',
    })
new_df_raw[-1]

In [ ]:
for x in balanced_socials.iterrows():
    new_df_raw.append({
        'author': x[1]['author'],
        'text': x[1]['text'],
        'source_id': x[1]['source_id'],
        'source_type': 'socials',
    })
new_df_raw[-1]

In [ ]:
concated_df = pd.DataFrame(new_df_raw)
concated_df.head()

In [ ]:
len(concated_df)

In [ ]:
lens = [
    {
        'df': 'socials_df',
        'authors': len(concated_df[concated_df['source_type'] == 'socials'].value_counts('author')),
    },
    {
        'df': 'ruslit_df',
        'authors': len(concated_df[concated_df['source_type'] == 'ruslit'].value_counts('author'))
    },
    {
        'df': 'scinews_df',
        'authors': len(concated_df[concated_df['source_type'] == 'nplus1'].value_counts('author'))
    }
]
lens = pd.DataFrame(lens)
fig, ax = plt.subplots(figsize=(16, 6))
ax = sns.barplot(x='df', y='authors',data=lens)
ax.bar_label(ax.containers[0])
ax.set_title('Author of every genre')
fig.tight_layout()

display(fig)

#mlflow.log_figure(fig, f"plots/genre_authors.png")

plt.close(fig)

In [ ]:
concated_df.to_csv('multigenres_df.csv', index=False)

In [ ]:
mlflow.log_artifact("multigenres_df.csv", artifact_path="data")

In [ ]:
mlflow.log_metric('dataset_size', len(concated_df))

In [ ]:
mlflow.end_run()